# Ultra Low Complexity Noise Suppression (ULCNet - Simplified)

This notebook implements an end-to-end simplified version of the ULCNet model
based on the paper "Ultra Low Complexity Deep Learning Based Noise Suppression".

Features:

- STFT + Power Law Compression
- Two-stage Network (CRN + CNN)
- Training + Inference
- Audio Reconstruction


## Install and import libraries


In [87]:

# Install dependencies (run once)
# !pip install torch torchaudio librosa soundfile tqdm


In [88]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from tqdm import tqdm
import os


## Setting up GPU


In [89]:

# GPU Setup and Verification
print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)

# Check and set GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.cuda.empty_cache()
    print("CUDA device initialized and cache cleared")
else:
    print("WARNING: CUDA not available. Using CPU (training will be slow)")

print("=" * 60)


GPU CONFIGURATION
Device: cuda
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA Version: 12.1
GPU Memory: 4.29 GB
CUDA device initialized and cache cleared


## Setup Variation Mode


In [ ]:
MODEL_VARIANT = "heavy"  # "heavy" or "light"

MODE = 2   # 0=vanilla, 1=stage1 only, 2=no compression

USE_STAGE2 = (MODE != 1)
USE_COMPRESSION = (MODE != 2)
ALPHA = 0.3 if USE_COMPRESSION else 1.0

## STFT and Power-Law Compression


In [91]:
# ---- Collate function for DataLoader --------------------------------------
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    """
    batch: list of tuples (mix_tensor, clean_tensor) with variable lengths
    Returns:
      mix_padded: [B, Lmax]
      clean_padded: [B, Lmax]
      lengths: tensor([L1, L2, ...])
    """
    mixes = [b[0].float() for b in batch]
    cleans = [b[1].float() for b in batch]
    lengths = torch.tensor([m.shape[0] for m in mixes], dtype=torch.long)

    mix_p = pad_sequence(mixes, batch_first=True)   # [B, Lmax]
    clean_p = pad_sequence(cleans, batch_first=True) # [B, Lmax]

    return mix_p, clean_p, lengths


In [92]:

def power_compress(x, alpha=0.3, use_compression=True):
    """Power-law compression."""
    if not use_compression:
        return x
    return torch.sign(x) * torch.abs(x) ** alpha

def power_decompress(x, alpha=0.3, use_compression=True):
    """Power-law decompression."""
    if not use_compression:
        return x
    return torch.sign(x) * torch.abs(x) ** (1 / alpha)


# ---- Batched STFT / ISTFT helpers ----------------------------------------
# Updated to support batched signals (B, L) or single (L,) - GPU optimized
def compute_stft(wav, n_fft=512, hop=256, win_length=512):
    """
    Compute STFT on GPU.
    wav: [B, L] or [L] tensor on GPU
    Returns: [B, F, T] or [F, T] complex tensor on same device
    """
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    # STFT stays on same device as input (GPU if input is on GPU)
    return torch.stft(wav, n_fft=n_fft, hop_length=hop, win_length=win_length,
                      return_complex=True, center=True)

def compute_istft(spec, n_fft=512, hop=256, win_length=512, length=None):
    """
    Compute ISTFT on GPU.
    spec: [B, F, T] or [F, T] complex tensor on GPU
    Returns: [B, L] or [L] waveform on same device as input
    """
    # ISTFT stays on same device as input (GPU if input is on GPU)
    return torch.istft(spec, n_fft=n_fft, hop_length=hop, win_length=win_length,
                       length=length, center=True)


## Stage 1: CRN (Magnitude Mask)


In [93]:

class CRNStage(nn.Module):
    def __init__(self, freq_bins=257):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, (1,3), padding=(0,1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, (1,3), padding=(0,1)),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, (1,3), padding=(0,1)),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.gru = nn.GRU(
            input_size=128*freq_bins,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Sequential(
            nn.Linear(512, freq_bins),
            nn.Sigmoid()
        )


    def forward(self, x):
        # x: [B,1,T,F]
        B,C,T,F = x.shape

        x = self.conv(x)

        x = x.permute(0,2,1,3)
        x = x.reshape(B, T, -1)

        x,_ = self.gru(x)

        mask = self.fc(x)

        return mask


## Stage 2: CNN (Complex Mask)


In [94]:

class CNNStage(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(2, 32, (1,3), padding=(0,1)),
            nn.ReLU(),

            nn.Conv2d(32, 32, (1,3), padding=(0,1)),
            nn.ReLU(),

            nn.Conv2d(32, 2, 1)
        )


    def forward(self, x):
        return self.net(x)


## Full ULCNet Model


In [95]:

class ULCNet(nn.Module):
    def __init__(self, freq_bins=257, alpha=0.3, mode=0):
        super().__init__()
        self.alpha = alpha
        self.mode = mode
        self.use_stage2 = (mode != 1)
        self.use_compression = (mode != 2)

        self.stage1 = CRNStage(freq_bins)
        self.stage2 = CNNStage() if self.use_stage2 else None

    def forward(self, mag, phase):
        mag = mag.unsqueeze(1)
        mask = self.stage1(mag)

        if not self.use_stage2:
            return mask, None, None

        yr = mask * torch.cos(phase)
        yi = mask * torch.sin(phase)
        feat = torch.stack([yr, yi], dim=1)

        cmask = self.stage2(feat)
        mr = cmask[:, 0]
        mi = cmask[:, 1]
        return mask, mr, mi

# Light Model Classes (Paper-like Implementation)


In [ ]:
class DepthwiseSeparableConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=(1, 3), padding=(0, 1), stride=(1, 1)):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_ch, in_ch, kernel_size=kernel_size, stride=stride,
            padding=padding, groups=in_ch, bias=False
        )
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.PReLU(out_ch)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.act(x)
        return x

In [ ]:
class ChannelwiseFeatureReorientation(nn.Module):
    """Approximation of the paper's channelwise feature reorientation."""
    def __init__(self, freq_bins=257, band_bins=48, band_hop=32):
        super().__init__()
        self.freq_bins = freq_bins
        self.band_bins = band_bins
        self.band_hop = band_hop
        # 8 bands of 48 bins with hop 32 -> 272 padded bins
        self.pad_to = band_bins + band_hop * 7

    def forward(self, x):
        # x: [B, T, F]
        b, t, f = x.shape
        if f < self.pad_to:
            x = F.pad(x, (0, self.pad_to - f))
        elif f > self.pad_to:
            x = x[..., :self.pad_to]
        bands = x.unfold(dimension=-1, size=self.band_bins, step=self.band_hop)
        # [B, T, 8, 48] -> [B, 8, T, 48]
        return bands.permute(0, 2, 1, 3).contiguous()

In [ ]:
class FrequencyBiGRU(nn.Module):
    """BiGRU applied along the frequency axis for each time frame."""
    def __init__(self, in_features, hidden_size=64):
        super().__init__()
        self.gru = nn.GRU(
            input_size=in_features,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        # x: [B, C, T, F]
        b, c, t, f = x.shape
        x = x.permute(0, 2, 3, 1).contiguous().view(b * t, f, c)  # [B*T, F, C]
        y, _ = self.gru(x)
        y = y.view(b, t, f, -1).permute(0, 3, 1, 2).contiguous()   # [B, 2H, T, F]
        return y

In [ ]:
class PaperLikeStage1(nn.Module):
    """Lightweight CRN-style magnitude mask estimator."""
    def __init__(self, freq_bins=257, band_bins=48, band_hop=32, num_bands=8):
        super().__init__()
        self.num_bands = num_bands
        self.reorient = ChannelwiseFeatureReorientation(freq_bins, band_bins, band_hop)

        # Paper-like depthwise-separable conv stack
        self.conv1 = DepthwiseSeparableConv2d(num_bands, 32)
        self.conv2 = DepthwiseSeparableConv2d(32, 64)
        self.conv3 = DepthwiseSeparableConv2d(64, 96)
        self.conv4 = DepthwiseSeparableConv2d(96, 128)

        # one gentle pooling step to keep the frequency resolution light
        self.pool = nn.MaxPool2d((1, 2), stride=(1, 2))

        # frequency-axis recurrent block
        self.fgru = FrequencyBiGRU(128, hidden_size=64)

        # pointwise projection after the recurrent block
        self.pw = nn.Conv2d(128, 64, kernel_size=1, bias=False)
        self.pw_bn = nn.BatchNorm2d(64)
        self.pw_act = nn.PReLU(64)

        # subband temporal GRU blocks (shared weights across subbands)
        self.band_gru = nn.GRU(
            input_size=64 * 3, hidden_size=64,
            num_layers=2, batch_first=True, bidirectional=True
        )

        # two FC layers for the intermediate magnitude mask
        self.fc1 = nn.Linear(num_bands * 128, 257)
        self.fc2 = nn.Linear(257, 257)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: [B, T, F]
        x = self.reorient(x)                 # [B, 8, T, 48]
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.pool(self.conv3(x))         # [B, 96, T, 24]
        x = self.conv4(x)                    # [B, 128, T, 24]
        x = self.fgru(x)                     # [B, 128, T, 24]
        x = self.pw_act(self.pw_bn(self.pw(x)))  # [B, 64, T, 24]

        b, c, t, f = x.shape
        band_width = f // self.num_bands     # 24 -> 3
        band_feats = []

        for band_idx in range(self.num_bands):
            seg = x[:, :, :, band_idx * band_width:(band_idx + 1) * band_width]  # [B,64,T,3]
            seg = seg.permute(0, 2, 3, 1).contiguous().view(b, t, -1)             # [B,T,192]
            y, _ = self.band_gru(seg)                                             # [B,T,128]
            band_feats.append(y)

        feat = torch.cat(band_feats, dim=-1)  # [B, T, 1024]
        mask = self.sigmoid(self.fc2(F.relu(self.fc1(feat))))
        return mask

In [ ]:
class PaperLikeStage2(nn.Module):
    """Tiny CNN used for the complex mask refinement stage."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=(1, 3), padding=(0, 1)),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=(1, 3), padding=(0, 1)),
            nn.ReLU(),
            nn.Conv2d(32, 2, kernel_size=1)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class ULCNet_Light(nn.Module):
    def __init__(self, freq_bins=257, alpha=0.3, mode=0):
        super().__init__()
        self.mode = mode
        self.alpha = alpha
        self.use_stage2 = (mode != 1)
        self.use_compression = (mode != 2)
        self.stage1 = PaperLikeStage1(freq_bins=freq_bins)
        self.stage2 = PaperLikeStage2()

    def forward(self, mag, phase):
        mag_mask = self.stage1(mag)

        if not self.use_stage2:
            return mag_mask, None, None

        yr = mag_mask * torch.cos(phase)
        yi = mag_mask * torch.sin(phase)
        feat = torch.stack([yr, yi], dim=1)

        cmask = self.stage2(feat)
        return mag_mask, cmask[:, 0], cmask[:, 1]

## Dataset Loader (DNS-Style Simulation)


In [96]:

class SimpleDataset(torch.utils.data.Dataset):

    def __init__(self, clean_files, noise_files, sr=16000):
        self.clean = clean_files
        self.noise = noise_files
        self.sr = sr


    def __len__(self):
        return len(self.clean)


    def __getitem__(self, idx):

        c,_ = librosa.load(self.clean[idx], sr=self.sr)
        n,_ = librosa.load(
            self.noise[idx],
            sr=self.sr
        )

        L = min(len(c), len(n))
        c = c[:L]
        mix = n[:L]

        return torch.tensor(mix), torch.tensor(c)


## Training Loop


In [97]:
# ---- Updated training loop (masks out padded samples) ---------------------
def train(model, loader, optimizer, device, n_fft=512, hop=256, alpha=0.3):
    model.train()
    total_loss = 0.0
    total_batches = 0

    for mix, clean, lengths in tqdm(loader):
        mix = mix.to(device)       # [B, Lmax] -> GPU
        clean = clean.to(device)   # [B, Lmax] -> GPU
        lengths = lengths.to(device)
        
        # Verify tensors are on GPU
        assert mix.device.type == device.type, f"Mix not on {device}"
        assert clean.device.type == device.type, f"Clean not on {device}"

        # 1) STFT (batched) - GPU operation
        X = compute_stft(mix, n_fft=n_fft, hop=hop)    # [B, F, T] complex on GPU
        
        # Verify STFT output is on GPU
        assert X.device.type == device.type, "STFT output not on GPU"

        # 2) Power-law compress real/imag - GPU operation
        Xr = power_compress(X.real, alpha=alpha, use_compression=model.use_compression)
        Xi = power_compress(X.imag, alpha=alpha, use_compression=model.use_compression)

        mag = torch.sqrt(Xr**2 + Xi**2).permute(0, 2, 1)
        phase = torch.atan2(Xi, Xr).permute(0, 2, 1)

        mag_mask, mr, mi = model(mag, phase)

        if model.use_stage2:
            mr = mr.permute(0, 2, 1)
            mi = mi.permute(0, 2, 1)
            est_r_c = mr * Xr - mi * Xi
            est_i_c = mr * Xi + mi * Xr
        else:
            # stage1 only path
            mag_hat = mag_mask * mag
            mag_hat = mag_hat.permute(0, 2, 1)
            phase_c = torch.atan2(Xi, Xr)
            est_r_c = mag_hat * torch.cos(phase_c)
            est_i_c = mag_hat * torch.sin(phase_c)

        est_r = power_decompress(est_r_c, alpha=alpha, use_compression=model.use_compression)
        est_i = power_decompress(est_i_c, alpha=alpha, use_compression=model.use_compression)

        # 7) Reconstruct time-domain signals (GPU operation)
        spec_est = torch.complex(est_r, est_i)  # [B, F, T] on GPU
        Lmax = mix.shape[1]
        out_wav = compute_istft(spec_est, n_fft=n_fft, hop=hop, win_length=n_fft, length=Lmax)  # [B, Lmax] on GPU
        
        # Verify ISTFT output is on GPU
        assert out_wav.device.type == device.type, "ISTFT output not on GPU"

        # 8) Compute MSE loss only over original lengths (GPU operation)
        batch_loss = 0.0
        total_valid = 0
        for i, l in enumerate(lengths):
            li = int(l.item())
            if li == 0:
                continue
            # use reduction='sum' to weight by length - GPU operation
            batch_loss += F.mse_loss(out_wav[i,:li], clean[i,:li], reduction='sum')
            total_valid += li

        if total_valid == 0:
            loss = torch.tensor(0.0, device=device)
        else:
            loss = batch_loss / total_valid

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(1, total_batches)

## Inference / Enhancement


In [99]:
# ---- Inference: accept (wav, original_length) and crop output (GPU optimized) ----
def enhance(model, wav, device, n_fft=512, hop=256, alpha=0.3):
    """
    Enhance noisy audio on GPU.
    wav: 1D torch tensor or numpy array (variable length)
    device: torch.device for computation
    Returns: enhanced audio clipped to original length
    """
    model.eval()
    if isinstance(wav, np.ndarray):
        wav = torch.tensor(wav, dtype=torch.float32, device=device)  # Create directly on device
    else:
        wav = wav.to(device)  # Move to device if already tensor

    orig_len = wav.shape[0]
    wav_b = wav.unsqueeze(0)  # [1, L] on device
    
    assert wav_b.device.type == device.type, "Input not on GPU"

    with torch.no_grad():
        # All operations now on GPU
        X = compute_stft(wav_b, n_fft=n_fft, hop=hop)  # [1, F, T] complex on GPU

        Xr = power_compress(X.real, alpha=model.alpha, use_compression=model.use_compression)
        Xi = power_compress(X.imag, alpha=model.alpha, use_compression=model.use_compression)

        mag = torch.sqrt(Xr**2 + Xi**2).permute(0, 2, 1)
        phase = torch.atan2(Xi, Xr).permute(0, 2, 1)

        mag_mask, mr, mi = model(mag, phase)
        if model.use_stage2:
            # ===== FULL MODEL =====
            mr = mr.permute(0, 2, 1)
            mi = mi.permute(0, 2, 1)

            est_r_c = mr * Xr - mi * Xi
            est_i_c = mr * Xi + mi * Xr

        else:
            # ===== STAGE 1 ONLY =====
            mag_hat = mag_mask * mag            # [B, T, F]
            mag_hat = mag_hat.permute(0, 2, 1)  # [B, F, T]

            phase_c = torch.atan2(Xi, Xr)

            est_r_c = mag_hat * torch.cos(phase_c)
            est_i_c = mag_hat * torch.sin(phase_c)

        est_r = power_decompress(est_r_c, alpha=model.alpha, use_compression=model.use_compression)
        est_i = power_decompress(est_i_c, alpha=model.alpha, use_compression=model.use_compression)

        spec = torch.complex(est_r, est_i)

        # ISTFT on GPU, length ensures proper cropping
        out = compute_istft(spec, n_fft=n_fft, hop=hop, win_length=n_fft, length=orig_len)  # [1, orig_len] on GPU
        out = out.squeeze(0).cpu().numpy()  # Move to CPU only for numpy conversion

    return out  # numpy array (on CPU)

## Example Usage


In [100]:
import os

running_on_kaggle = False
if 'KAGGLE_URL_BASE' in os.environ:
    running_on_kaggle = True

In [101]:
if running_on_kaggle:
    # training params
    clean_dir = "/kaggle/input/datasets/sayuksh/denoising-audio-collection/clean_trainset_56spk_wav/clean_trainset_56spk_wav"
    noisy_dir = "/kaggle/input/datasets/sayuksh/denoising-audio-collection/noisy_trainset_56spk_wav/noisy_trainset_56spk_wav"

    # testing params
    test_file = "/kaggle/input/datasets/sayuksh/denoising-audio-collection/noisy_testset_wav/noisy_testset_wav/p232_101.wav"
    output_location = "enhanced_output.wav"
else:
    # training params
    clean_dir = "./clean_trainset_wav"
    noisy_dir = "./noisy_trainset_wav"

    # testing params
    test_file = "./app2_data/noisy/p232_003.wav"
    output_location = "enhanced_output.wav"

In [ ]:

# Example (requires your own wav files)

# Initialize GPU device (from earlier GPU setup)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create model and move to GPU
if MODEL_VARIANT == "heavy":
    model = ULCNet(mode=MODE, alpha=ALPHA).to(device)
else:
    model = ULCNet_Light(mode=MODE, alpha=ALPHA).to(device)
print(f"Model moved to {device}")

print("Variant:", MODEL_VARIANT)
print("Mode:", model.mode)
print("Stage2:", model.use_stage2)
print("Compression:", model.use_compression)

# Create optimizer
opt = torch.optim.Adam(model.parameters(), lr=4e-4)

# Prepare dataset
clean_files = []
noise_files = []

# clean files append
for filename in os.listdir(clean_dir):
    if filename.endswith(".wav"):
        clean_files.append(os.path.join(clean_dir, filename))
        
# noise files append
for filename in os.listdir(noisy_dir):
    if filename.endswith(".wav"):
        noise_files.append(os.path.join(noisy_dir, filename))
  
# Uncomment for quick testing
clean_files = clean_files[:5]
noise_files = noise_files[:5]

print(f"Found {len(clean_files)} clean files and {len(noise_files)} noise files")
    
# ---- DataLoader usage: pass collate_fn -----------------------------------
dataset = SimpleDataset(clean_files, noise_files, sr=16000)
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0   # IMPORTANT: Keep at 0 for CUDA on Windows
)

print("DataLoader created successfully")
print(f"Total batches per epoch: {len(loader)}")


Using device: cuda
Model moved to cuda
Found 5 clean files and 5 noise files
DataLoader created successfully
Total batches per epoch: 2


In [ ]:
# Train with GPU monitoring
from importlib.resources import path
import time

print("="*60)
print("TRAINING ON GPU")
print("="*60)

for epoch in range(10):
    start_time = time.time()
    loss = train(model, loader, opt, device)
    epoch_time = time.time() - start_time
    
    print(f"Epoch {epoch:2d} | Loss: {loss:.6f} | Time: {epoch_time:.2f}s")
    
    model_path = f"ulcnet_mode_{model.mode}_epoch{epoch}.pth"
    
    # Save model
    torch.save({
    "state_dict": model.state_dict(),
    "mode": model.mode,
    "alpha": model.alpha,
    "variant": MODEL_VARIANT
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"         GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")
        
print("="*60)
print("Training Complete!")
print("="*60)


TRAINING ON GPU


100%|██████████| 2/2 [00:03<00:00,  1.62s/it]


Epoch  0 | Loss: 0.003478 | Time: 3.24s
         GPU Memory - Allocated: 1.45GB, Reserved: 6.60GB


100%|██████████| 2/2 [00:03<00:00,  1.62s/it]


Epoch  1 | Loss: 0.002597 | Time: 3.24s
         GPU Memory - Allocated: 1.45GB, Reserved: 6.61GB


...

Epoch  7 | Loss: 0.002265 | Time: 4.05s
         GPU Memory - Allocated: 1.45GB, Reserved: 6.61GB


100%|██████████| 2/2 [00:03<00:00,  1.84s/it]


Epoch  8 | Loss: 0.002461 | Time: 3.68s
         GPU Memory - Allocated: 1.45GB, Reserved: 6.61GB


100%|██████████| 2/2 [00:03<00:00,  1.82s/it]


Epoch  9 | Loss: 0.002331 | Time: 3.65s
         GPU Memory - Allocated: 1.45GB, Reserved: 6.61GB
Training Complete!


In [105]:
# Enhance with GPU
print("Enhancing audio on GPU...")
wav, _ = librosa.load(test_file, sr=16000)
start_time = time.time()
out = enhance(model, torch.tensor(wav), device)  # Use GPU for enhancement
inference_time = time.time() - start_time

sf.write(output_location, out, 16000)
print(f"Enhanced audio saved to {output_location}")
print(f"Inference time: {inference_time:.3f}s")


Enhancing audio on GPU...
Enhanced audio saved to enhanced_output.wav
Inference time: 0.179s


## Test set


In [106]:
if running_on_kaggle:
    # test set directory location
    test_noise_dir = "/kaggle/input/datasets/sayuksh/denoising-audio-collection/noisy_testset_wav/noisy_testset_wav"
    test_clean_dir = "/kaggle/input/datasets/sayuksh/denoising-audio-collection/clean_testset_wav/clean_testset_wav"
else:
    # test set directory location
    test_noise_dir = "./noisy_testset_wav/"
    test_clean_dir = "./clean_testset_wav/"

In [107]:
# load filenames
test_noisy_files = []
test_clean_files = []

for filename in os.listdir(test_noise_dir):
    if filename.endswith(".wav"):
        test_noisy_files.append(os.path.join(test_noise_dir, filename))
for filename in os.listdir(test_clean_dir):
    if filename.endswith(".wav"):
        test_clean_files.append(os.path.join(test_clean_dir, filename))
        
# uncomment for quick testing
test_noisy_files = test_noisy_files[:5]
test_clean_files = test_clean_files[:5]

In [108]:
# define error function (GPU optimized)
def compute_mse_on_test_set(model, noisy_files, clean_files, device):
    """Compute MSE on test set using GPU for all computations."""
    model.eval()
    total_mse = 0.0
    total_samples = 0

    with torch.no_grad():
        for nfile, cfile in zip(noisy_files, clean_files):
            noisy_wav, _ = librosa.load(nfile, sr=16000)
            clean_wav, _ = librosa.load(cfile, sr=16000)

            # Enhance on GPU
            enhanced_wav = enhance(model, torch.tensor(noisy_wav), device)

            L = min(len(enhanced_wav), len(clean_wav))
            # Compute MSE
            mse = np.mean((enhanced_wav[:L] - clean_wav[:L])**2)

            total_mse += mse * L
            total_samples += L

    overall_mse = total_mse / total_samples if total_samples > 0 else 0.0
    return overall_mse


In [109]:
# ouput error on test set
mse_error = compute_mse_on_test_set(model, test_noisy_files, test_clean_files, device)
print("MSE on test set:", mse_error)

MSE on test set: 0.0038429523345970156


In [110]:
# Comprehensive GPU performance check
print("\n" + "="*60)
print("GPU PERFORMANCE SUMMARY")
print("="*60)
print(f"Device: {device}")
print(f"Model location: {next(model.parameters()).device}")
print(f"Total model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Mode: {model.mode}")
print(f"Use stage 2: {model.use_stage2}")
print(f"Use compression: {model.use_compression}")
print(f"Alpha: {model.alpha}")

if torch.cuda.is_available():
    print(f"\nGPU Memory:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"  Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    
    # Memory breakdown
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1e9:.2f} GB")
        print(f"  Used: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
        
print("\n✓ All computations are running on GPU")
print("="*60)



GPU PERFORMANCE SUMMARY
Device: cuda
Model location: cuda:0
Total model parameters: 51,091,267
Mode: 2
Use stage 2: True
Use compression: False
Alpha: 1.0

GPU Memory:
  Allocated: 1.45 GB
  Reserved:  6.61 GB

GPU 0: NVIDIA GeForce RTX 3050 Laptop GPU
  Total Memory: 4.29 GB
  Used: 1.45 GB

✓ All computations are running on GPU


## Notes

This is a faithful but simplified implementation for learning and experimentation.

For production, full subband GRUs and channelwise reorientation should be added.
